# Trace the Ace — Runner

**This notebook is a thin, stable wrapper. All logic lives in the `traceace` package on GitHub.**

The loop: fix code locally → `git push` → re-run **one cell** here. `run()` calls `sync()` first,
so every cell picks up the latest code **without a runtime reset**.

> ⚠️ **Read the RUNTIME banner above each cell before running it.** Compute units burn while the
> runtime is *connected*, not while it computes — an idle attached GPU is the biggest waste vector.
> CPU is free and covers most of this project. The tier guard will refuse CPU tasks on a GPU.


### 🖥️ RUNTIME: **CPU + High RAM** — ~0 units/hr · run once per runtime
Mounts Drive, clones/pulls the repo, defines `sync()` and `run()`.

In [ ]:
import os, sys, subprocess, traceback
from google.colab import drive
drive.mount('/content/drive')

REPO_URL   = "https://github.com/meteorboyF/Trace-the-Ace--meteorboyF.git"
REPO_DIR   = "/content/Trace-the-Ace"
DRIVE_ROOT = "/content/drive/MyDrive/trace-the-ace"
BRANCH     = "main"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

def sync(branch=BRANCH, install=False):
    """git pull -> purge cached modules -> re-import. Returns the package."""
    subprocess.run(["git","-C",REPO_DIR,"fetch","--all","--quiet"], check=True)
    subprocess.run(["git","-C",REPO_DIR,"checkout",branch,"--quiet"], check=True)
    subprocess.run(["git","-C",REPO_DIR,"reset","--hard",f"origin/{branch}","--quiet"], check=True)
    src = os.path.join(REPO_DIR, "src")
    if src not in sys.path:
        sys.path.insert(0, src)
    if install:
        subprocess.run([sys.executable,"-m","pip","install","-q","-r",
                        os.path.join(REPO_DIR,"requirements-colab.txt")], check=True)
    for m in [m for m in list(sys.modules) if m == "traceace" or m.startswith("traceace.")]:
        del sys.modules[m]
    import traceace
    traceace.configure(repo_dir=REPO_DIR, drive_root=DRIVE_ROOT)
    sha = subprocess.run(["git","-C",REPO_DIR,"rev-parse","--short","HEAD"],
                         capture_output=True, text=True).stdout.strip()
    print(f"synced @ {sha}")
    return traceace

def run(task, **kw):
    """sync, then run a task. Never kills the kernel on failure."""
    tc = sync()
    try:
        return tc.tasks.run(task, **kw)
    except Exception:
        traceback.print_exc()
        return None

tc = sync(install=True)   # install=True only needed once per runtime

---
## 1 · Data staging and EDA

### 🖥️ RUNTIME: **CPU + High RAM** — ~0 units/hr · est. 5–10 min (first run)
Stages `raw.zip` from Drive to the local SSD and extracts it **locally** (Drive is a FUSE
mount at ~100–300 ms per file op — never iterate it). No-op if already staged.
Do **not** run on GPU.

In [ ]:
run("data.ingest")          # normalize suffixed filenames by CONTENT SHAPE
run("data.consolidate")     # 22,821 transcript CSVs -> one parquet (once)

### 🖥️ RUNTIME: **CPU + High RAM** — ~0 units/hr · est. 8 min
`eda.transcripts` is the most important measurement in the project — it drives the
architecture decision. `eda.inference_budget` turns it into a go/no-go table.

In [ ]:
run("eda.overview")
run("eda.transcripts")          # exact token/char distributions (~8 min, progress bar)
run("eda.inference_budget")     # go/no-go vs the 6-hour A100 cap

---
## 2 · Folds and baselines

### 🖥️ RUNTIME: **CPU + High RAM** — ~0 units/hr · est. <1 min
`cv.build` generates session-grouped folds **once** and persists them; every model reuses them.
`baseline.lo_only` is **the bar every real model must clear** — it is the organizers' stated
anti-goal implemented as a baseline.

In [ ]:
run("cv.build")
run("baseline.prior")       # the floor    (expected logloss ~0.609)
run("baseline.lo_only")     # THE BAR      (no transcript information at all)

### 🖥️ RUNTIME: **CPU + High RAM** — ~0 units/hr · est. 8 min
**Run this before making any feature-block decision.** At 35K rows the paired delta SD is
~5e-4, so single-seed differences below ~1e-3 are noise. These tasks report mean ± SD across
repeated fold assignments.

In [ ]:
run("evaluate.repeated")            # headline score with error bars
run("interpret.ablation_repeated")  # PAIRED leave-one-block-out, mean ± SD

---
## 3 · Feature blocks  (all CPU — the cheap ladder)

### 🖥️ RUNTIME: **CPU + High RAM** — ~0 units/hr · est. 20 min total
Cached to parquet; reruns are free (loud `CACHE HIT`). Pass `force=True` to recompute.

`features.lo_alignment` is the **LO-conditioned** block — the only one that can separate two
responses from the same session, which is 43% of the label variance.

In [ ]:
run("features.structural")      # ~2.5 min
run("features.linguistic")      # ~6 min  (incl. ASR disfluency markers)
run("features.temporal")        # ~3 min  (robust statistics)
run("features.lo_alignment")    # ~7 min  (KEY MOMENTS, response-level)

---
## 4 · Model, calibration, research artifacts

### 🖥️ RUNTIME: **CPU + High RAM** — ~0 units/hr · est. 3 min
LightGBM on CPU beats GPU at this data size. Calibration matters as much as ranking for log loss.

In [ ]:
run("model.gbdt")
run("calibrate.fit", experiment="model.gbdt")
run("evaluate.report", experiment="model.gbdt")
run("interpret.report", experiment="model.gbdt")   # figures -> artifacts/figures/

### 🖥️ RUNTIME: **CPU + High RAM** — ~0 units/hr · est. 10 min
Ablation gives each feature block's marginal contribution — so claims are about *what*
mattered, not merely *that* something worked. Log negative results too.

In [ ]:
run("interpret.ablation")

---
## 5 · Tutoring-move taxonomy

### 🖥️ RUNTIME: **CPU + High RAM** — ~0 units/hr · est. 5 min  (heuristic backend)
Free rule-based labelling — run this first and inspect the distribution.

In [ ]:
run("annotate.moves", backend="heuristic")
run("model.move_classifier")

### ⚡ RUNTIME: **L4 GPU** — ~5 units/hr · est. ~8 h ≈ **40 units** · run **ONCE**
LLM annotation of a stratified training sample. **DEV-TIME ONLY** — by ADR-004 no generative
model ever enters the submission; this only produces training labels for the cheap classifier.

**Smoke it first.** Check the sample output before committing 40 units.

In [ ]:
run("annotate.moves", backend="vllm", subsample=200)   # ← SMOKE FIRST
# run("annotate.moves", backend="vllm", shutdown_after=True)   # the real run (~40 units)

---
## 6 · Semantic LO-alignment  (the highest-value experiment)

### ⚡ RUNTIME: **L4 GPU** — ~5 units/hr · est. **35–52 min ≈ 3–4 units** · run **ONCE**

Lexical (TF-IDF) matching is the current ceiling: an objective phrased *"multiply a 2-digit
number by a 1-digit number"* is taught as *"so what's seven threes?"* — near-zero word overlap,
near-total semantic overlap.

Model: `BAAI/bge-small-en-v1.5` (**MIT**, 33M params, 512 ctx, built for query→passage retrieval).
Projection measured from local CPU throughput: ~610K windows.

**Smoke first.** If the smoke cell errors, fix locally and re-run — do not run the full cell.

In [ ]:
run("features.window_embeddings", subsample=500)   # ← SMOKE FIRST (~2 min)
# run("features.window_embeddings", shutdown_after=True)   # full run, ~35-52 min

### 🖥️ RUNTIME: **CPU + High RAM** — ~0 units/hr · est. 10 min
Once the vectors are cached, everything downstream is CPU. `features.content` pools the top-k
window vectors and PCA-reduces them — *what was said*, as opposed to how much.

In [ ]:
run("features.lo_alignment", backend="embedding", force=True)
run("features.content")                      # pooled + PCA-48, WHAT was said
run("interpret.ablation_repeated")           # re-rank blocks WITH error bars

---
## 7 · Submission

### 🖥️ RUNTIME: **CPU + High RAM** — ~0 units/hr · est. 2 min
`verify` is deliberately paranoid: only **three full submissions per week** (~15 attempts left).
A loud verify is worth more than a clever model.

In [ ]:
run("submission.build", experiment="model.gbdt")
run("submission.smoke")                 # runs main.py as a subprocess
run("submission.verify", smoke=True)    # raises on ANY violation

### ⚡ RUNTIME: **A100 80GB** — ~12 units/hr · est. 20 min ≈ **4 units**
**Only for final timing validation** against the real inference hardware, right before a real
submission. Nothing else needs an A100.

In [ ]:
run("submission.smoke", allow_waste=True)   # A100 timing parity check

---
## 8 · Session close-out

### 🖥️ RUNTIME: **any** — run this at the END of every session
Syncs artifacts to Drive (tarred — one big file, never a directory walk), prints the budget
report, then disconnects so billing stops.

In [ ]:
run("maintenance.sync_artifacts")   # tars artifacts + runs, single large write to Drive
run("budget.report")                # spend by task and tier vs the 733-unit balance
run("docs.build")                   # regenerate docs/EXPERIMENTS.md from runs/

# Disconnect so an attached GPU stops burning units:
# from google.colab import runtime; runtime.unassign()